In [1]:
import pyspark.sql.functions as f
from pyspark.sql import SparkSession
from pyspark import SparkConf

In [2]:
parameters = {
    "spark.driver.maxResultSize": "3g",
    "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.sql.execution.arrow.pyspark.enabled": True,

    # https://docs.kedro.org/en/stable/integrations/pyspark_integration.html#tips-for-maximising-concurrency-using-threadrunner
    "spark.scheduler.mode": "FAIR",
    "spark.driver.extraJavaOptions": "-Djava.security.manager=allow",
    "spark.executor.extraJavaOptions": "-Djava.security.manager=allow",

    "spark.sql.legacy.parquet.nanosAsLong": True,
    "spark.driver.memory": "40g",
}

spark_conf = SparkConf().setAll(parameters.items())
spark = SparkSession.builder.appName('New').enableHiveSupport().config(conf=spark_conf).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/28 17:13:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
ftr_transactions = spark.read.parquet("../data/04_feature/ftr_transactions")

In [ ]:
ftr_transactions.columns

['_id',
 '_observ_end_dt',
 'month_total_sales',
 'month_net_sales',
 'month_total_profit',
 'month_total_cost',
 'month_total_amount_discounts',
 'month_total_qty_discounts',
 'month_avg_percent_discount',
 'month_distinct_skus_sold',
 'month_distinct_branches',
 'month_distinct_product_categories',
 'month_distinct_transactions',
 'month_total_discount_percentual',
 'month_avg_order',
 'month_avg_skus_per_order',
 'month_net_sales_category_ac',
 'month_total_profit_category_ac',
 'month_total_cost_category_ac',
 'month_total_amount_discounts_category_ac',
 'month_total_qty_discounts_category_ac',
 'month_net_sales_category_additives',
 'month_total_profit_category_additives',
 'month_total_cost_category_additives',
 'month_total_amount_discounts_category_additives',
 'month_total_qty_discounts_category_additives',
 'month_net_sales_category_air_filter',
 'month_total_profit_category_air_filter',
 'month_total_cost_category_air_filter',
 'month_total_amount_discounts_category_air_filt

25/05/28 17:14:04 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
25/05/29 06:38:00 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1035484 ms exceeds timeout 120000 ms
25/05/29 06:38:00 WARN SparkContext: Killing executors is not supported by current scheduler.
25/05/29 06:54:19 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.Rp

## Fix Churn features

In [5]:
from pyspark.sql import Window

In [9]:
working_ids = [
    "00085C8C-BC2A-4DBA-B083-C56A47C27525__8DA35057-0961-4744-A97F-499F45E47BF0",
    "000BECC8-C481-4836-AFB4-A546BE802A4F__0C4F8AE6-E0FA-4DDA-A809-85A6032CEA8B",
    "000F803A-79A2-4666-B548-257B376B20C1__F45146CF-52D5-49C5-8BE1-9F3BE0C3AAD0",
    "00002C41-103F-4481-BF8C-ED3290CED3AB__CDBEB6FF-6A71-498D-9468-17A850FF851D",
    "00004275-76A3-48E7-8FA3-C4B5C1BAE8BF__D80BA2ED-C9A2-479E-98AF-90ABFA8A3046", # Fix dropping decrease
    "00004E48-9244-48B6-AA95-561E2870E465__ABDEC947-486B-412B-819C-094571535E85",
    "00005D42-1D81-4D5F-9894-ED4D312E8462__0640CD72-3355-4D12-91E9-27F5B1B3C20D",
    "0000AD06-2D25-4EC8-BCCE-75B65ADB6110__2F8AABB7-5CDB-4C4D-9739-FD387AD42F2B",
    "0000F4D0-43F2-47BB-9746-A966A37060FD__7A5771BE-AA18-4E32-8A5B-9FB956B88A1A", # Fix with mpd std drop 2 transaction
    "00025DDD-B44D-4016-BFA9-F9948C6D3EC1__C9784263-B912-404A-A9F1-DFFA570547B6",
]

In [11]:
import pandas as pd

In [12]:
agos_cols = ["mobile", "plate_number", "car_make", "car_model", "language", "current_mileage", "forecasted_mileage", "days_to_change_oil", "is_due_this_month"]
exploring_cols = [
    "_id", "_observ_end_dt",
    "last_trx_current_mileage", "has_oil", "has_oil_synthetic",
    "customer_current_mileage", "customer_transaction_dt", "customer_last_current_mileage",
    "oil_mileage_per_day", "oil_avg_mileage_per_day",
    "oil_synthetic_mileage_per_day", "oil_synthetic_avg_mileage_per_day",
    "customer_mileage_per_day", "customer_avg_mileage_per_day",

    'oil_last_transaction_dt',
    'oil_synthetic_last_transaction_dt',
    'customer_last_transaction_dt',
    'customer_days_since_last_trx',
    'customer_days_until_change',
    'customer_forecast_mileage',
    'customer_is_due',
    'customer_mileage_when_due',
    'oil_days_since_last_trx',
    'oil_days_until_change',
    'oil_forecast_mileage',
    'oil_is_due',
    'oil_mileage_when_due',
    'oil_synthetic_days_since_last_trx',
    'oil_synthetic_days_until_change',
    'oil_synthetic_forecast_mileage',
    'oil_synthetic_is_due',
    'oil_synthetic_mileage_when_due',


    'estimated_days_to_change_mineral_oil',
    'estimated_days_to_change_synthetic_oil',
    'days_since_oil_last_transactions',
    'days_since_oil_synthetic_last_transactions',
    'days_until_mineral_oil_change',
    'days_until_synthetic_oil_change',

    'is_active',
    'is_churn',
    'target_churn_1',
]

selected_cols = [
    "_id", "_observ_end_dt",
    "customer_current_mileage",
    "customer_last_current_mileage",
    'customer_last_transaction_dt',
    # "oil_mileage_per_day",
    # "oil_avg_mileage_per_day",
    # "oil_synthetic_mileage_per_day",
    # "oil_synthetic_avg_mileage_per_day",
    'oil_last_transaction_dt',
    'oil_synthetic_last_transaction_dt',
    "customer_mileage_per_day",
    "customer_avg_mileage_per_day",
    'customer_days_since_last_trx',
    'customer_days_until_change',
    'customer_forecast_mileage',
    'customer_is_due',
    'customer_mileage_when_due',

    'oil_days_since_last_trx',
    'oil_is_due',
    'oil_synthetic_days_since_last_trx',
    'oil_synthetic_is_due',


    'estimated_days_to_change_mineral_oil',
    'estimated_days_to_change_synthetic_oil',

    'days_since_oil_last_transactions',
    'days_since_oil_synthetic_last_transactions',
    # 'days_until_mineral_oil_change',
    # 'days_until_synthetic_oil_change',

    'is_active',
    'is_churn',
    'target_churn_1',
]


In [17]:
ftr_transactions.filter(
    f.year("_observ_end_dt") == 2024
).groupBy(
    "_observ_end_dt",
).agg(
    f.sum(
        f.when(
            f.col("month_net_sales") > 0,
            1
        ).otherwise(0)
    )
).show(20, truncate=False)

ftr_transactions.filter(
    f.year("_observ_end_dt") == 2024
).groupBy(
    "_observ_end_dt",
).agg(
    f.sum(
        f.when(
            f.col("month_net_sales") > 0,
            1
        ).otherwise(0)
    ).alias("active_per_month")
).select(
    f.mean("active_per_month")
).show(20, truncate=False)

+--------------+------------------------------------------------------+
|_observ_end_dt|sum(CASE WHEN (month_net_sales > 0) THEN 1 ELSE 0 END)|
+--------------+------------------------------------------------------+
|2024-09-30    |241728                                                |
|2024-03-31    |214312                                                |
|2024-11-30    |198428                                                |
|2024-02-29    |220782                                                |
|2024-12-31    |196642                                                |
|2024-07-31    |232652                                                |
|2024-01-31    |229954                                                |
|2024-05-31    |232650                                                |
|2024-08-31    |234469                                                |
|2024-10-31    |215228                                                |
|2024-06-30    |223700                                          

+---------------------+
|avg(active_per_month)|
+---------------------+
|221014.25            |
+---------------------+



In [19]:
ftr_transactions.filter(
    f.year("_observ_end_dt") == 2024
).filter(
    f.col("month_net_sales") > 0
).groupBy(
    "_observ_end_dt",
).agg(
    f.mean("month_net_sales").alias("avg_month_net_sales")
).show(20, truncate=False)

ftr_transactions.filter(
    f.year("_observ_end_dt") == 2024
).filter(
    f.col("month_net_sales") > 0
).groupBy(
    "_observ_end_dt",
).agg(
    f.mean("month_net_sales").alias("avg_month_net_sales")
).select(
    f.mean("avg_month_net_sales")
).show(20, truncate=False)

+--------------+-------------------+
|_observ_end_dt|avg_month_net_sales|
+--------------+-------------------+
|2024-09-30    |266.76548930344575 |
|2024-03-31    |275.46582869963845 |
|2024-11-30    |294.7488550773746  |
|2024-02-29    |265.4269218941291  |
|2024-12-31    |286.02757199225607 |
|2024-07-31    |272.83701848427734 |
|2024-01-31    |271.8944151162017  |
|2024-05-31    |272.12129967032837 |
|2024-08-31    |275.0306159497138  |
|2024-10-31    |278.91400476674085 |
|2024-06-30    |270.893081963858   |
|2024-04-30    |270.2155526962399  |
+--------------+-------------------+



+------------------------+
|avg(avg_month_net_sales)|
+------------------------+
|275.0283879678503       |
+------------------------+



In [21]:
ftr_transactions.filter(
    f.year("_observ_end_dt") == 2025
).filter(
    f.col("month_net_sales") > 0
).groupBy(
    "_observ_end_dt",
).agg(
    f.mean("month_net_sales").alias("avg_month_net_sales")
).show(20, truncate=False)


ftr_transactions.filter(
    f.year("_observ_end_dt") == 2025
).filter(
    f.col("month_net_sales") > 0
).groupBy(
    "_observ_end_dt",
).agg(
    f.mean("month_net_sales").alias("avg_month_net_sales")
).select(
    f.mean("avg_month_net_sales")
).show(20, truncate=False)

+--------------+-------------------+
|_observ_end_dt|avg_month_net_sales|
+--------------+-------------------+
|2025-02-28    |273.40332426219516 |
|2025-03-31    |288.5681251813073  |
|2025-05-31    |261.81701683040905 |
|2025-01-31    |292.83173905852397 |
|2025-04-30    |275.0276631398405  |
+--------------+-------------------+



+------------------------+
|avg(avg_month_net_sales)|
+------------------------+
|278.32957369445523      |
+------------------------+



In [ ]:
ftr_transactions.filter(
    f.col("_observ_end_dt") == "2025-05-31"
).withColumn(
    "delta_mileage",
    f.col("customer_forecast_mileage") - f.col("customer_last_current_mileage")
).withColumn(
    "delta_mileage",
    f.col("customer_forecast_mileage") - f.col("customer_last_current_mileage")
).select(*selected_cols).show(2, truncate=False)

+---------------------+--------------+------------------------+-----------------------------+----------------------------+-----------------------+---------------------------------+------------------------+----------------------------+----------------------------+--------------------------+-------------------------+---------------+-------------------------+-----------------------+----------+---------------------------------+--------------------+------------------------------------+--------------------------------------+--------------------------------+------------------------------------------+---------+--------+--------------+
|_id                  |_observ_end_dt|customer_current_mileage|customer_last_current_mileage|customer_last_transaction_dt|oil_last_transaction_dt|oil_synthetic_last_transaction_dt|customer_mileage_per_day|customer_avg_mileage_per_day|customer_days_since_last_trx|customer_days_until_change|customer_forecast_mileage|customer_is_due|customer_mileage_when_due|oil_day

In [13]:
ftr_transactions.filter(
    f.col("_id") == "5205SLJ__966545078777"
).select(*selected_cols).orderBy("_id", "_observ_end_dt").show(60, truncate=False)

+---------------------+--------------+------------------------+-----------------------------+----------------------------+-----------------------+---------------------------------+------------------------+----------------------------+----------------------------+--------------------------+-------------------------+---------------+-------------------------+-----------------------+----------+---------------------------------+--------------------+------------------------------------+--------------------------------------+--------------------------------+------------------------------------------+---------+--------+--------------+
|_id                  |_observ_end_dt|customer_current_mileage|customer_last_current_mileage|customer_last_transaction_dt|oil_last_transaction_dt|oil_synthetic_last_transaction_dt|customer_mileage_per_day|customer_avg_mileage_per_day|customer_days_since_last_trx|customer_days_until_change|customer_forecast_mileage|customer_is_due|customer_mileage_when_due|oil_day